In [ ]:
# !pip install pytrends
# !pip install nltk

In [ ]:
# Import neccessary libraries

import pandas as pd
import numpy as np
import re
import time
import matplotlib.pyplot as plt
import pytrends
import seaborn as sns

from pytrends.request import TrendReq
# import nltk

pd.set_option('future.no_silent_downcasting', True)
sns.set()

In [ ]:
# nltk.download("punkt")
# nltk.download("punkt_tab")

### Fetching and Comparing Interest Over Time

In [ ]:
pytrends = TrendReq(hl="en-US", tz=60) # Initialize TrendReq()

keywords = ["DANGOTE CEMENT", "GUARANTY TRUST BANK", "ZENITH BANK"]

timeframe = "today 5-y" # "today 12-m"

# pytrends.build_payload(kw_list=keywords, timeframe=timeframe)
pytrends.build_payload(kw_list=keywords, timeframe=timeframe, geo="NG") # Set to Nigeria Geographical Area

interest_over_time_df = pytrends.interest_over_time()
# interest_over_time_df = interest_over_time_df.fillna(False)
interest_over_time_df.query("isPartial == False", inplace=True)

n = len(interest_over_time_df.columns) # Numeber of Columns
interest_over_time_df = interest_over_time_df.iloc[:, :n-1]
interest_over_time_df.to_csv("../outputs/SVI_Interest_Over_Time_NG.csv", sep=",")

In [ ]:
interest_over_time_df.head() # Overview of the dataset

In [ ]:
# Plotting the interest over time
plt.figure(figsize=(16, 8))
for keyword in keywords:
    sns.lineplot(
        x=interest_over_time_df.index, 
        y=interest_over_time_df[keyword],
        label = keyword,
        linewidth=2
    )
plt.xlabel("Date")
plt.ylabel("Trends Index")
plt.title("Interest Over Time")
plt.legend(loc="upper right")
plt.grid(True)
plt.tight_layout()
plt.savefig("../outputs/SVI_Interest_Over_Time_NG.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
avg_SVI = interest_over_time_df.mean()  # Average SVI
std_SVI = interest_over_time_df.std()   # Std Dev SVI
max_SVI = interest_over_time_df.max()   # Maximum SVI
min_SVI = interest_over_time_df.min()   # Minimum SVI

summary_stats = pd.DataFrame({
    "Keyword": interest_over_time_df.columns,
    "Mean SVI": avg_SVI,
    "Std Dev SVI": std_SVI,
    "Max SVI (Peak)": max_SVI,
    "Min SVI": min_SVI
})
# summary["Average SVI"] = avg_SVI
# summary["Standard Deviation SVI"] = std_SVI
summary_stats.reset_index(drop=True, inplace=True)
summary_stats.to_csv("../outputs/SVI_Summary_Statistics.csv")
print("Summary Statistics for Search Volume Index:")
display(summary_stats)

#### Correlation of Each Keywords

In [ ]:
##K Keywords Correlations
corr_interest_over_time = interest_over_time_df.corr()
corr_interest_over_time.to_csv("../outputs/SVI_Corr_Interest_Over_Time_NG.csv", sep=",")
corr_interest_over_time

**Interest Over Time:**

> The Google Trends data shows that **Zenith Bank consistently dominates search interest in Nigeria** compared to Dangote Cement and Guaranty Trust Bank. From 2021 to 2026, Zenith Bank’s trend line remains significantly higher, with noticeable peaks that suggest periods of heightened public attention—likely tied to product launches, service issues, or major announcements. In contrast, Dangote Cement and Guaranty Trust Bank maintain relatively low and stable search volumes, indicating less frequent spikes in public interest.

This suggests that **consumer-facing financial institutions attract more consistent online attention** than industrial companies like Dangote Cement, which may only trend during specific events (e.g., corporate news or policy changes).

### Analyzing Interest by Region

In [ ]:
pytrends.build_payload(kw_list=keywords, timeframe=timeframe, geo="NG")  # Set to Nigeria Geographical Area
interest_by_region_df = pytrends.interest_by_region()
interest_by_region_df.index.name = "REGION"
interest_by_region_df.to_csv("../outputs/SVI_Interest_By_Region_NG.csv", sep=",")

#### Sort by DANGOTE CEMENT

In [ ]:
interest_by_region_df.sort_values(by="DANGOTE CEMENT", ascending=False).head()

**Note:**

> **Dangote Cement** registers modest interest, with its highest values in Kogi and Benue.

#### Sort by GUARANTY TRUST BANK

In [ ]:
interest_by_region_df.sort_values(by="GUARANTY TRUST BANK", ascending=False).head()

In [ ]:
min_, max_ = interest_by_region_df["GUARANTY TRUST BANK"].min(), interest_by_region_df["GUARANTY TRUST BANK"].max()
print("Minimum search score:", min_)
print("Maximum search score:", max_)

**Note:**

> **Guaranty Trust Bank** appears weaker in regional searches, often scoring below 5 in most states.

#### Sort by ZENITH BANK

In [ ]:
interest_by_region_df.sort_values(by="ZENITH BANK", ascending=False).head()

**Note:**

> **Zenith Bank** shows overwhelming dominance in states such as Yobe, Taraba, and Anambra, with values above 90 in relative interest.

---

**Interest by Region:**

Regional analysis highlights strong variation across Nigerian states:

- **Zenith Ban**k shows overwhelming dominance in states such as Yobe, Taraba, and Anambra, with values above 90 in relative interest.

- **Dangote Cement** registers modest interest, with its highest values in Kogi and Benue.

- **Guaranty Trust Bank** appears weaker in regional searches, often scoring below 5 in most states.

### Analyzing Related Queries (Top and Rising)

In [ ]:
pytrends.build_payload(kw_list=keywords, timeframe=timeframe, geo="NG") # Set to Nigeria Geographical Area
related_query_dict = pytrends.related_queries()

#### For Dangote Cement

In [ ]:
display(related_query_dict["DANGOTE CEMENT"]["top"])

In [ ]:
display(related_query_dict["DANGOTE CEMENT"]["rising"])

#### For Guaranty Trust Bank

In [ ]:
display(related_query_dict["GUARANTY TRUST BANK"]["top"])

In [ ]:
display(related_query_dict["GUARANTY TRUST BANK"]["rising"])

#### For Zenith Bank

In [ ]:
top = related_query_dict["ZENITH BANK"]["top"]
top.rename(columns={"query": "Query", "value": "Value"}, inplace=True)
top.to_csv("../outputs/SVI_Zenith_Top_Related_Queries.csv", sep=",")
display(top.head())

**Note:**
> Top queries include "zenith bank code", "zenith bank customer care", and “zenith bank internet”.

In [ ]:
rising = related_query_dict["ZENITH BANK"]["rising"]
rising.rename(columns={"query": "Query", "value": "Value"}, inplace=True)
top.to_csv("../outputs/SVI_Zenith_Rising_Related_Queries.csv", sep=",")
display(rising.head())

**Note:**
> Rising queries emphasize digital access issues such as "zenith bank customer care number 24 hours" and "zenith bank network today".
---

**Summary of Related Queries**

For **Zenith Bank**, related queries are heavily directed toward **customer service and digital banking tools**:

- Top queries include "zenith bank code", "zenith bank customer care", and “zenith bank internet”.

- Rising queries emphasize digital access issues such as "zenith bank customer care number 24 hours" and "zenith bank network today".

However, Dangote Cement and Guaranty Trust Bank return no top and rising queries

---

## Overall Interpretation


Overall, the analysis shows:

- **Zenith Bank** dominates both national and regional search interest, driven by customer service and digital banking queries.

- **Dangote Cement** and **Guaranty Trust Bank** attract far less attention, with interest concentrated in specific regions.

- The data underscores the **importance of digital accessibility and customer support** in shaping public search behavior for Nigerian banks.

---